# Creating a Sentence classification model using BERT and logistic model.

This jupyter-notebook is different from the previous one, here I am exploring Fine-Tuning

In [41]:
import numpy as np
import pandas as pd
import torch
import transformers as ppb
import warnings
warnings.filterwarnings('ignore')

In [42]:
!pip install scikit-learn

# 1. Loading the dataset

In [43]:
df = pd.read_csv("https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv", delimiter='\t', header=None)

In [44]:
df.head()

,0,1
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0
3,this is a visually stunning rumination on love...,1
4,jonathan parker 's bartleby should have been t...,1


this time, as we have to do finetuning, we will use the full dataset

In [45]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [46]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

#3. Using Trainer API of the transformers library

In [47]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("
bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


This will issue a warning about some of the pretrained weights not being used and some weights being randomly initialized. That’s because we are throwing away the pretraining head of the BERT model to replace it with a classification head which is randomly initialized. We will fine-tune this model on our task, transferring the knowledge of the pretrained model to it (which is why doing this is called transfer learning).

#4. The Trainer API()

Then, to define our Trainer, we will need to instantiate a TrainingArguments. This class contains all the hyperparameters we can tune for the Trainer or the flags to activate the different training options it supports. Let’s begin by using all the defaults, the only thing we then have to provide is a directory in which the checkpoints will be saved:

In [48]:
from transformers import TrainingArguments

training_args = TrainingArguments("test_trainer")

In [49]:
from torch.utils.data import Dataset

class SST2Dataset(Dataset):

    def __init__(self, dataframe, tokenizer):

        self.texts = dataframe[0].tolist()
        self.labels = dataframe[1].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }

In [50]:
train_dataset = SST2Dataset(
    train_df,
    tokenizer
)

eval_dataset = SST2Dataset(
    eval_df,
    tokenizer
)

In [51]:
# now we can instantiate a trainer:

from transformers import Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset=eval_dataset
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.432205
1000,0.292143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#5. Evaluation

In [56]:
!pip install evaluate

In [57]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits , axis=-1)
  return metric.compute(predictions=predictions,references = labels)

In [58]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    compute_metrics=compute_metrics
)

trainer.evaluate()

{'eval_loss': 0.4467220902442932,
 'eval_model_preparation_time': 0.0026,
 'eval_accuracy': 0.9140173410404624,
 'eval_runtime': 13.6785,
 'eval_samples_per_second': 101.181,
 'eval_steps_per_second': 12.648}